In [4]:
import gc
import sys
import os
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from scipy import signal
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, TensorDataset


# =========================================================
# 1. CONFIGURATION
# =========================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("===== DEBUG PYTHON / CUDA =====")
print("Python executable:", sys.executable)
print("Torch file:", torch.__file__)
print("Torch version:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this Python process. "
        "This script is probably using a different Python interpreter/environment."
    )

device = torch.device("cuda:0")

print("Device utilisé :", device)
print("GPU utilisée :", torch.cuda.get_device_name(0))

if torch.cuda.device_count() > 1:
    print("GPU 1 disponible :", torch.cuda.get_device_name(1))

print("================================")

path = r"C:\Users\al6268\Downloads\filtered_3_labels.parquet"

WINDOW_SIZE = 50
STRIDE = 25
BATCH_SIZE = 1024 if device.type == "cpu" else 4096

VAE_EPOCHS = 50
JUDGE_EPOCHS = 30

LATENT_DIM = 32
LABEL_EMB_DIM = 16


# =========================================================
# 2. CHARGEMENT ET FENÊTRAGE DES DONNÉES
# =========================================================

X = []
labels = []
groups = []

parquet_file = pq.ParquetFile(path)

print(f"Nombre de paquets à traiter : {parquet_file.num_row_groups}")

for i in range(parquet_file.num_row_groups):
    chunk = parquet_file.read_row_group(
        i,
        columns=[
            "acc_x",
            "acc_y",
            "acc_z",
            "global_activity_id",
            "dataset",
            "subject_id",
            "session_id",
        ],
    ).to_pandas()

    # Magnitude accéléromètre
    acc_mag = np.sqrt(
        chunk["acc_x"].values ** 2
        + chunk["acc_y"].values ** 2
        + chunk["acc_z"].values ** 2
    ).astype(np.float32)

    chunk["acc_mag"] = acc_mag
    chunk.drop(columns=["acc_x", "acc_y", "acc_z"], inplace=True)

    # Important : ne pas mélanger différentes sessions/sujets
    for group_key, group_df in chunk.groupby(["dataset", "subject_id", "session_id"]):
        sig = group_df["acc_mag"].values.astype(np.float32)
        acts = group_df["global_activity_id"].values

        if len(sig) < WINDOW_SIZE:
            continue

        for j in range(0, len(sig) - WINDOW_SIZE + 1, STRIDE):
            window = sig[j : j + WINDOW_SIZE]
            label = Counter(acts[j : j + WINDOW_SIZE]).most_common(1)[0][0]

            X.append(window)
            labels.append(label)
            groups.append(str(group_key))

    del chunk
    gc.collect()

    if (i + 1) % 5 == 0:
        print(f"Paquet {i + 1}/{parquet_file.num_row_groups} terminé...")

X = np.asarray(X, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)

print(f"Chargement terminé. Nombre de fenêtres : {len(X)}")


# =========================================================
# 3. MAPPING DES LABELS
# =========================================================

unique_labels = sorted(np.unique(labels))
label_mapping = {old: new for new, old in enumerate(unique_labels)}
inv_label_mapping = {new: old for old, new in label_mapping.items()}

labels_mapped = np.asarray([label_mapping[l] for l in labels], dtype=np.int64)

n_classes = len(unique_labels)

print("Mapping des labels :")
for old, new in label_mapping.items():
    print(f"Ancien label {old} -> Nouveau label {new}")

print(f"Nombre de classes : {n_classes}")


# =========================================================
# 4. SPLIT TRAIN / VAL / TEST PAR SESSION
# =========================================================
# Cela évite que des fenêtres très similaires de la même session
# se retrouvent à la fois en train et en test.

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(X, labels_mapped, groups=groups))

X_train_val = X[train_val_idx]
y_train_val = labels_mapped[train_val_idx]
groups_train_val = groups[train_val_idx]

X_test = X[test_idx]
y_test = labels_mapped[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2222, random_state=SEED)
train_idx_rel, val_idx_rel = next(
    gss_val.split(X_train_val, y_train_val, groups=groups_train_val)
)

X_train = X_train_val[train_idx_rel]
y_train = y_train_val[train_idx_rel]

X_val = X_train_val[val_idx_rel]
y_val = y_train_val[val_idx_rel]

print(f"Train : {len(X_train)} fenêtres")
print(f"Val   : {len(X_val)} fenêtres")
print(f"Test  : {len(X_test)} fenêtres")


# =========================================================
# 5. NORMALISATION
# =========================================================
# Important : calculer mean/std uniquement sur train,
# puis appliquer aux trois splits.

mean = np.mean(X_train)
std = np.std(X_train)

print(f"Mean train : {mean:.6f}")
print(f"Std train  : {std:.6f}")

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

# Vérifications de sécurité
for name, arr in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    print(
        f"{name} | min={arr.min():.3f} max={arr.max():.3f} "
        f"mean={arr.mean():.3f} std={arr.std():.3f}"
    )

    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contient des NaN ou Inf.")


# Shape PyTorch : (batch, channels, length)
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)


# =========================================================
# 6. MODÈLES
# =========================================================

class ConditionalVAE(nn.Module):
    def __init__(self, n_classes, latent_dim=32, label_emb_dim=16):
        super().__init__()

        self.n_classes = n_classes
        self.latent_dim = latent_dim
        self.label_emb_dim = label_emb_dim

        self.label_emb = nn.Embedding(n_classes, label_emb_dim)

        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 50 -> 25

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),  # 25 -> 12

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        encoder_out_dim = 128 * 12

        self.fc_mu = nn.Linear(encoder_out_dim + label_emb_dim, latent_dim)
        self.fc_logvar = nn.Linear(encoder_out_dim + label_emb_dim, latent_dim)

        self.dec_input = nn.Linear(latent_dim + label_emb_dim, 128 * 13)

        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(
                128,
                64,
                kernel_size=3,
                stride=2,
                padding=1,
            ),  # 13 -> 25
            nn.ReLU(),

            nn.ConvTranspose1d(
                64,
                32,
                kernel_size=5,
                stride=2,
                padding=2,
                output_padding=1,
            ),  # 25 -> 50
            nn.ReLU(),

            nn.Conv1d(32, 1, kernel_size=5, padding=2),
        )

    def encode(self, x, y):
        h = self.encoder(x)
        y_emb = self.label_emb(y)
        h_cond = torch.cat([h, y_emb], dim=1)

        mu = self.fc_mu(h_cond)
        logvar = self.fc_logvar(h_cond)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-10, max=10)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y):
        y_emb = self.label_emb(y)
        z_cond = torch.cat([z, y_emb], dim=1)

        h = self.dec_input(z_cond)
        h = h.view(-1, 128, 13)

        return self.decoder(h)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, y)

        return recon, mu, logvar

    def sample(self, n_samples, class_id, device):
        self.eval()

        y = torch.full(
            size=(n_samples,),
            fill_value=class_id,
            dtype=torch.long,
            device=device,
        )

        z = torch.randn(n_samples, self.latent_dim, device=device)

        with torch.no_grad():
            samples = self.decode(z, y)

        return samples


class DeepConvLSTM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.transpose(1, 2)
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])


class CNNSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class MLPSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),

            nn.Linear(WINDOW_SIZE, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)


# =========================================================
# 7. ENTRAÎNEMENT VAE STABLE
# =========================================================

def train_vae(model, loader, val_loader=None, epochs=50):
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    beta = 0.001

    for ep in range(epochs):
        model.train()

        total_loss = 0.0
        total_mse = 0.0
        total_kld = 0.0
        total_seen = 0

        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()

            recon, mu, logvar = model(xb, yb)

            logvar = torch.clamp(logvar, min=-10, max=10)

            mse = nn.functional.mse_loss(recon, xb, reduction="mean")

            kld_per_sample = -0.5 * torch.sum(
                1 + logvar - mu.pow(2) - logvar.exp(),
                dim=1,
            )
            kld = kld_per_sample.mean()

            loss = mse + beta * kld

            if not torch.isfinite(loss):
                print("Loss non finie détectée pendant le VAE.")
                print(f"MSE: {mse.item()}")
                print(f"KLD: {kld.item()}")
                print(f"xb min/max: {xb.min().item()} / {xb.max().item()}")
                print(f"mu min/max: {mu.min().item()} / {mu.max().item()}")
                print(f"logvar min/max: {logvar.min().item()} / {logvar.max().item()}")
                return

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            batch_size = xb.size(0)
            total_loss += loss.item() * batch_size
            total_mse += mse.item() * batch_size
            total_kld += kld.item() * batch_size
            total_seen += batch_size

        train_loss = total_loss / total_seen
        train_mse = total_mse / total_seen
        train_kld = total_kld / total_seen

        if val_loader is not None:
            val_loss, val_mse, val_kld = evaluate_vae(model, val_loader, beta)
            print(
                f"VAE Epoch {ep + 1:03d} | "
                f"Train Loss: {train_loss:.5f} | "
                f"MSE: {train_mse:.5f} | "
                f"KLD: {train_kld:.5f} || "
                f"Val Loss: {val_loss:.5f} | "
                f"Val MSE: {val_mse:.5f} | "
                f"Val KLD: {val_kld:.5f}"
            )
        else:
            print(
                f"VAE Epoch {ep + 1:03d} | "
                f"Loss: {train_loss:.5f} | "
                f"MSE: {train_mse:.5f} | "
                f"KLD: {train_kld:.5f}"
            )


def evaluate_vae(model, loader, beta=0.001):
    model.eval()

    total_loss = 0.0
    total_mse = 0.0
    total_kld = 0.0
    total_seen = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            recon, mu, logvar = model(xb, yb)

            logvar = torch.clamp(logvar, min=-10, max=10)

            mse = nn.functional.mse_loss(recon, xb, reduction="mean")

            kld_per_sample = -0.5 * torch.sum(
                1 + logvar - mu.pow(2) - logvar.exp(),
                dim=1,
            )
            kld = kld_per_sample.mean()

            loss = mse + beta * kld

            batch_size = xb.size(0)
            total_loss += loss.item() * batch_size
            total_mse += mse.item() * batch_size
            total_kld += kld.item() * batch_size
            total_seen += batch_size

    return (
        total_loss / total_seen,
        total_mse / total_seen,
        total_kld / total_seen,
    )


# =========================================================
# 8. ENTRAÎNEMENT DES JUGES
# =========================================================

def train_judge(model, train_loader, val_loader=None, epochs=30):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for ep in range(epochs):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            if not torch.isfinite(loss):
                print("Loss non finie détectée pendant un juge.")
                return

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            preds = torch.argmax(logits, dim=1)

            batch_size = xb.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (preds == yb).sum().item()
            total_seen += batch_size

        train_loss = total_loss / total_seen
        train_acc = total_correct / total_seen

        if val_loader is not None:
            val_loss, val_acc = evaluate_judge(model, val_loader)
            print(
                f"Epoch {ep + 1:03d} | "
                f"Train Loss: {train_loss:.5f} | "
                f"Train Acc: {train_acc:.4f} || "
                f"Val Loss: {val_loss:.5f} | "
                f"Val Acc: {val_acc:.4f}"
            )
        else:
            print(
                f"Epoch {ep + 1:03d} | "
                f"Train Loss: {train_loss:.5f} | "
                f"Train Acc: {train_acc:.4f}"
            )


def evaluate_judge(model, loader):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            preds = torch.argmax(logits, dim=1)

            batch_size = xb.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (preds == yb).sum().item()
            total_seen += batch_size

    return total_loss / total_seen, total_correct / total_seen


# =========================================================
# 9. ENTRAÎNEMENT PRINCIPAL
# =========================================================

vae_model = ConditionalVAE(
    n_classes=n_classes,
    latent_dim=LATENT_DIM,
    label_emb_dim=LABEL_EMB_DIM,
).to(device)

print("\nEntraînement VAE conditionnel")
train_vae(vae_model, train_loader, val_loader, epochs=VAE_EPOCHS)

judges = [
    DeepConvLSTM(n_classes).to(device),
    CNNSimple(n_classes).to(device),
    MLPSimple(n_classes).to(device),
]

for judge in judges:
    print(f"\nEntraînement Juge : {judge.__class__.__name__}")
    train_judge(judge, train_loader, val_loader, epochs=JUDGE_EPOCHS)


# =========================================================
# 10. ANALYSE PHYSIQUE
# =========================================================

def analyse_gait_physique(sig, fs=25):
    mag = sig.flatten().astype(np.float32)
    mag = mag - np.mean(mag)

    corr = np.correlate(mag, mag, mode="full")[len(mag) - 1 :]
    corr = corr / (np.max(corr) + 1e-7)

    min_lag = int(0.4 * fs)
    max_lag = int(1.2 * fs)

    if len(corr) > max_lag:
        peak_idx = np.argmax(corr[min_lag:max_lag]) + min_lag
        step_time = peak_idx / fs
        return corr[peak_idx], step_time

    return 0.0, 1e-7


def denormalize_signal(x):
    return x * (std + 1e-8) + mean


def full_expert_validation(old_label_id, nom_activite):
    vae_model.eval()

    if old_label_id not in label_mapping:
        raise ValueError(
            f"Le label original {old_label_id} n'existe pas dans les données. "
            f"Labels disponibles : {list(label_mapping.keys())}"
        )

    mapped_label_id = label_mapping[old_label_id]

    with torch.no_grad():
        gen_tensor = vae_model.sample(
            n_samples=30,
            class_id=mapped_label_id,
            device=device,
        )

    gen_np_norm = gen_tensor.cpu().numpy()
    gen_np = denormalize_signal(gen_np_norm)

    full_sig = gen_np.flatten()

    score_gait, step_t = analyse_gait_physique(gen_np[0, 0, :])

    f, t_spec, Sxx = signal.spectrogram(full_sig, fs=25)

    fig = plt.figure(figsize=(15, 10))

    plt.subplot(3, 1, 1)
    plt.plot(full_sig[:1000])
    plt.title(f"Génération conditionnelle : {nom_activite} | Label original {old_label_id}")

    plt.subplot(3, 2, 3)
    plt.plot(full_sig[250:375], marker=".")
    plt.title("Zoom signal généré")

    plt.subplot(3, 2, 4)
    plt.pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-7), shading="gouraud")
    plt.title("Spectre")

    ax_txt = plt.subplot(3, 1, 3)
    ax_txt.axis("off")

    txt = "VERDICTS JUGES :\n"

    for judge in judges:
        judge.eval()

        with torch.no_grad():
            logits = judge(gen_tensor)
            probs = torch.softmax(logits, dim=1).mean(0)

            pred_mapped_label = torch.argmax(probs).item()
            pred_old_label = inv_label_mapping[pred_mapped_label]

            confidence = probs.max().item() * 100

            is_correct = pred_old_label == old_label_id

            txt += (
                f"- {judge.__class__.__name__}: "
                f"Label prédit {pred_old_label} | "
                f"Conf {confidence:.1f}% | "
                f"{'✅' if is_correct else '❌'}\n"
            )

    txt += f"\nGAIT: {score_gait:.2f} | Cadence: {60 / step_t:.1f} steps/min"

    ax_txt.text(
        0,
        0.5,
        txt,
        fontsize=12,
        family="monospace",
        verticalalignment="center",
    )

    plt.tight_layout()
    plt.show()


# Exemple : génération conditionnée sur l'ancien label 1
full_expert_validation(1, "Marche")

===== DEBUG PYTHON / CUDA =====
Python executable: c:\Users\al6268\Desktop\text_to_imu_generation\.venv\Scripts\python.exe
Torch file: c:\Users\al6268\Desktop\text_to_imu_generation\.venv\Lib\site-packages\torch\__init__.py
Torch version: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: False
CUDA device count: 0
CUDA_VISIBLE_DEVICES: None


RuntimeError: CUDA is not available in this Python process. This script is probably using a different Python interpreter/environment.